# 🔥 Notebook 2: Cascading Failure — Breaker vs. No Breaker

Picture two services:

```
  clients ──► Service A ──► Service B (sick: slow + erroring)
```

B is slow. Every request to A blocks waiting for B. A's thread pool fills up. Now A itself looks dead to *its* callers. Congratulations: one small failure took down the whole stack. This is a **cascading failure**.

A circuit breaker between A and B turns that minute-long outage into a handful of fast errors. Let's measure it.

## 🛠️ Setup

```bash
cd 05-microservices/circuit-breaker
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

In [ ]:
import time, threading, statistics
from concurrent.futures import ThreadPoolExecutor

SLOW_B_DELAY = 1.0  # B hangs a full second before failing

def slow_b():
    time.sleep(SLOW_B_DELAY)
    raise RuntimeError('B timeout')

# --- A without a breaker: every call pays the full 1s timeout ---
def a_no_breaker():
    t0 = time.time()
    try:
        slow_b()
    except Exception:
        pass
    return time.time() - t0  # per-request latency

# --- A thread-safe breaker ---
class CB:
    def __init__(self, thr=5, reset=10.0):
        self.state = 'CLOSED'
        self.failures = 0
        self.opened_at = 0.0
        self.thr = thr
        self.reset = reset
        self.lock = threading.Lock()

    def call(self, fn):
        with self.lock:
            if self.state == 'OPEN':
                if time.time() - self.opened_at < self.reset:
                    raise RuntimeError('fast-fail: circuit OPEN')
                self.state = 'HALF_OPEN'
        try:
            r = fn()
            with self.lock:
                self.state = 'CLOSED'; self.failures = 0
            return r
        except Exception:
            with self.lock:
                self.failures += 1
                if self.state == 'HALF_OPEN' or self.failures >= self.thr:
                    self.state = 'OPEN'
                    self.opened_at = time.time()
            raise

cb = CB(thr=3, reset=10.0)

def a_with_breaker():
    t0 = time.time()
    try:
        cb.call(slow_b)
    except Exception:
        pass
    return time.time() - t0


### Simulate load — with a *realistic arrival pattern*

Two details make this demo honest, and both are easy to get wrong:

1. **A small worker pool.** In a real server the pool is finite. When threads are stuck
   on B, the pool is what runs out.
2. **Requests arrive over time**, not all at once, and we measure **end-to-end**
   latency — *queue wait plus service time*. Timing only the handler hides the queue,
   and the queue is where a cascading failure actually lives.

In [ ]:
SLA_MS   = 300     # A promises its own callers a response within 300 ms
ARRIVALS = 24      # requests A receives
RATE_S   = 0.05    # one new request every 50 ms  (= 20 req/s incoming)
WORKERS  = 4       # A's thread pool

def run_load(handler):
    """Feed requests in at a steady rate and record end-to-end latency each one saw."""
    records = []
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        t0 = time.time()
        futures = []
        for _ in range(ARRIVALS):
            arrived = time.time()
            futures.append((arrived, ex.submit(lambda: (handler(), time.time()))))
            time.sleep(RATE_S)
        for arrived, f in futures:
            _svc, done = f.result()
            records.append(done - arrived)     # queue wait + service time
        total = time.time() - t0
    return total, records

def report(label, total, lats):
    ms = sorted(l * 1000 for l in lats)
    p50 = ms[len(ms) // 2]
    p99 = ms[int(0.99 * (len(ms) - 1))]
    breached = sum(1 for l in ms if l > SLA_MS)
    print(f'{label:14s} wall={total:5.2f}s  p50={p50:8.1f}ms  p99={p99:8.1f}ms  '
          f'blew the {SLA_MS}ms SLA: {breached:2d}/{len(ms)}')

print(f'B is sick: hangs {SLOW_B_DELAY:.0f}s, then errors.')
print(f'{ARRIVALS} requests arrive at A at {1/RATE_S:.0f} req/s. A has {WORKERS} workers.')
print(f"A's own SLA to its callers is {SLA_MS} ms.\n")

total, lats = run_load(a_no_breaker)
report('no breaker', total, lats)

cb = CB(thr=3, reset=10.0)     # fresh breaker so the comparison is fair
total, lats = run_load(a_with_breaker)
report('with breaker', total, lats)


**Reading the numbers.** Watch p50 and the SLA column, not the wall clock.

- *no breaker*: **every** request pays B's full 1 s timeout. A only has 4 workers but
  requests keep arriving at 20/s, so the queue grows without bound and p50 climbs into
  **multiple seconds** — most of which is time spent *waiting for a worker*, not
  waiting for B. Every single request blows A's SLA. To A's own callers, **A is down**,
  even though nothing is wrong with A. That is the cascade.
- *with breaker*: the first handful of requests still pay the timeout — that's how the
  breaker learns. After 3 failures it trips, the queue drains in one burst, and
  everything arriving afterwards is answered immediately. p50 drops by roughly an order
  of magnitude and a large share of requests get back inside the SLA.

Two honest caveats about that second row:

1. Requests that were **already queued** when the breaker tripped still ate the
   timeout. A breaker limits the damage going forward; it can't refund latency
   that was already spent. (Shrinking that window is what a low `thr` and a
   **timeout** in front of the breaker buy you.)
2. The requests still **fail**. A breaker does not make a broken dependency work. It
   converts a slow, thread-eating failure into a fast, cheap one — which is the
   difference between "one dependency has an issue" and "the whole site is down".
   Turning the fast failure into something *useful* is the next section's job.

## Bonus: add a fallback

Fast-failing is better than hanging, but a **fallback** is better than an error. Common fallbacks:
- cached value (even a bit stale),
- default / empty result,
- degraded feature ("recommendations unavailable" instead of 500).

The breaker stays the same; we just catch its fast-fail and return something useful.

In [ ]:
CACHE = {'recommendations': ['popular-item-1', 'popular-item-2']}
cb2 = CB(thr=3, reset=10.0)

def get_recommendations():
    try:
        return cb2.call(slow_b)  # would return fresh personalised recs
    except Exception:
        return CACHE['recommendations']  # graceful degradation

for i in range(6):
    t0 = time.time()
    result = get_recommendations()
    print(f'  req {i}: {time.time()-t0:5.2f}s  →  {result}')


## Counting the *right* errors

A breaker's whole job is to detect "the dependency is unwell". A `404` or a `422` is
not the dependency being unwell — it is **your request** being wrong. Counting client
errors means a burst of bad input from one caller trips the breaker for everybody.

Below, the same 20 calls run through two breakers: one that counts every exception,
and one that only counts *server-side* failures.

In [ ]:
class HttpError(Exception):
    def __init__(self, status):
        super().__init__(f'HTTP {status}')
        self.status = status

def is_dependency_failure(exc):
    """Only these mean 'the downstream is unwell'."""
    if isinstance(exc, (TimeoutError, ConnectionError)):
        return True
    if isinstance(exc, HttpError):
        return exc.status >= 500 or exc.status == 429
    return False

class SelectiveCB(CB):
    """Same breaker, but non-dependency errors propagate WITHOUT being counted."""
    def call(self, fn):
        with self.lock:
            if self.state == 'OPEN':
                if time.time() - self.opened_at < self.reset:
                    raise RuntimeError('fast-fail: circuit OPEN')
                self.state = 'HALF_OPEN'
        try:
            r = fn()
        except Exception as e:
            if not is_dependency_failure(e):
                raise                      # caller's fault → don't blame the downstream
            with self.lock:
                self.failures += 1
                if self.state == 'HALF_OPEN' or self.failures >= self.thr:
                    self.state = 'OPEN'; self.opened_at = time.time()
            raise
        with self.lock:
            self.state = 'CLOSED'; self.failures = 0
        return r

# A healthy service. One buggy client sends 20 malformed requests -> 20x HTTP 422.
def healthy_but_bad_request():
    raise HttpError(422)

naive     = CB(thr=3, reset=10.0)
selective = SelectiveCB(thr=3, reset=10.0)

for breaker in (naive, selective):
    for _ in range(20):
        try: breaker.call(healthy_but_bad_request)
        except Exception: pass

print(f'counts every exception : state={naive.state}      '
      '← healthy dependency is now cut off for EVERY caller')
print(f'counts only 5xx/429/etc: state={selective.state}    '
      '← breaker stays closed; the bad client just keeps getting 422s')

### 🧠 Tips for production
- **Per-dependency breakers.** One breaker per downstream; don't share across services.
- **Fallbacks must be cheap.** Don't call another flaky service from your fallback.
- **Monitor state transitions.** Emit a metric every time the breaker opens/half-opens/closes. Dashboards love this.
- **Don't trip on user errors.** 4xx responses are the *caller's* fault; don't count them as failures.
- **Combine carefully with retries.** Retries inside a breaker are fine; retries *around* a breaker can hammer the probe. See Notebook 3.